In [3]:
## Load Environment Variables
import os  
from dotenv import load_dotenv
load_dotenv()
print("Environment variables loaded")

Environment variables loaded


In [4]:
## Imports from Langchain
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List


In [6]:
## Custom Pre-Processor Function to Process the PDF
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""
    def __init__(self,chunk_size=1000,chunk_overlap=100):
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.text_splitter=RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],

        )

    def process_pdf(self,pdf_path:str)->List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Laod PDF

        loader=PyMuPDFLoader(pdf_path)
        pages=loader.load()

        # Process each page

        processed_chunks=[]

        for page_num,page in enumerate(pages):
            # clean text
            cleaned_text=self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)
                }]
            )
            
            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())
        
        # Fix common PDF extraction issues
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        
        return text

In [7]:
preprocessor=SmartPDFProcessor()

In [8]:
preprocessor

In [9]:
## Process a PDF with Pre-Processor Function and Create Chunks
try:
    smart_chunks=preprocessor.process_pdf("Data/FUNDAMENTAL-RIGHTS.pdf")  # document path
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

Processed into 42 smart chunks

Sample chunk metadata:
  producer: Acrobat Distiller 7.0 (Windows)
  creator: PageMaker 7.0
  creationdate: 2015-10-13T14:44:01+05:30
  source: Data/FUNDAMENTAL-RIGHTS.pdf
  file_path: Data/FUNDAMENTAL-RIGHTS.pdf
  total_pages: 15
  format: PDF 1.6
  title: 
  author: winxp
  subject: 
  keywords: 
  moddate: 2015-10-13T14:44:11+05:30
  trapped: 
  modDate: D:20151013144411+05'30'
  creationDate: D:20151013144401+05'30'
  page: 1
  chunk_method: smart_pdf_processor
  char_count: 1715


In [10]:
# Print the chunks
print(smart_chunks[0])
print("-----------------------------")
print(smart_chunks[5])
print("-----------------------------")
print(smart_chunks[11])
print("-----------------------------")
print(smart_chunks[41])


page_content='PART III FUNDAMENTAL RIGHTS General 12. In this Part, unless the context otherwise requires, “the State’’ includes the Government and Parliament of India and the Government and the Legislature of each of the States and all local or other authorities within the territory of India or under the control of the Government of India. 13. (1) All laws in force in the territory of India immediately before the commencement of this Constitution, in so far as they are inconsistent with the provisions of this Part, shall, to the extent of such inconsistency, be void. (2) The State shall not make any law which takes away or abridges the rights conferred by this Part and any law made in contravention of this clause shall, to the extent of the contravention, be void. (3) In this article, unless the context otherwise requires,— (a) “law” includes any Ordinance, order, bye-law, rule, regulation, notification, custom or usage having in the territory of India the force of law; (b) “laws in f

In [11]:
## Import required modules (LATEST imports)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
## Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",   # Huggingface embeddings model
    model_kwargs={"device": "cpu"},                        # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4551.34it/s]


In [13]:
## Create FAISS Vector Store from smart_chunks
vectorstore = FAISS.from_documents(
    documents=smart_chunks,
    embedding=embeddings
)
print(f"\n Embeddings are created and stored in memory.")


 Embeddings are created and stored in memory.


In [14]:
## Save FAISS Index to Local Disk

FAISS_INDEX_PATH = "faiss_index"

vectorstore.save_local(FAISS_INDEX_PATH)

print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [15]:
## Load FAISS Index Later

loaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 42 vectors


In [16]:
## Quick Similarity Test
query = "What is this document about?"
docs = loaded_vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])


--- Result 1 ---
cl. (1) (with retrospective effect). THE CONSTITUTION OF INDIA (Part III.—Fundamental Rights.—Arts. 30—31A.) 15

--- Result 2 ---
15. (1) The State shall not discriminate against any citizen on grounds only of religion, race, caste, sex, place of birth or any of them. (2) No citizen shall, on grounds only of religion, race, caste, sex, place of birth or any of them, be subject to any disability, liability, restriction or condi

--- Result 3 ---
or abridges any of the rights conferred by, any provisions of this Part, and notwithstanding any judgment, decree or order of any court or Tribunal to the contrary, each of the said Acts and Regulations shall, subject to the power of any competent Legislature to repeal or amend it, continue in force


In [17]:
load_dotenv()    # Load environment

True

In [18]:
os.getenv("GROQ_API_KEY")      

'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'

In [19]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")    

In [20]:
## Define GROQ LLM
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="groq:llama-3.1-8b-instant"   # ✅ valid Groq model
)

In [21]:
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FA42AA0830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FA42AA1400>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [22]:
llm.invoke("Explain what is LangChain in one paragraph.")  # Test Query to LLM

AIMessage(content='LangChain is an open-source Python framework for building and training large language models, particularly those based on the transformer architecture. It provides a set of tools and libraries designed to streamline the process of creating, training, and deploying AI models, making it easier for developers to integrate and interact with these models in their applications. LangChain allows users to define custom workflows and pipelines for training and deploying models, and also includes tools for data loading, data preprocessing, and model evaluation. Its primary goal is to simplify the development process for building conversational AI models, allowing developers to focus on the logic and functionality of their applications rather than the underlying machine learning infrastructure.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 45, 'total_tokens': 176, 'completion_time': 0.211447477, 'completion_tokens_details'

In [23]:
# Define Prompt Template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [24]:
# Define Retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FA243AF4D0>, search_kwargs={'k': 3})

In [26]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [27]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Create RAG Chain

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [28]:
rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FA243AF4D0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object a

In [29]:
response = rag_chain.invoke(
    "Which articles states about Cultural and Educational Rights?"
)

print(response)

Article 29. 

It is mentioned in Document 1 (Source: Data/FUNDAMENTAL-RIGHTS.pdf): 
"Cultural and Educational Rights 29. (1) Any section of the citizens residing in the territory of India or any part thereof having a distinct language, script or culture of its own shall have the right to conserve the same. (2) No citizen shall be denied admission into any educational institution maintained by the State or receiving aid out of State funds on grounds only of religion, race, caste, language or any of them."


In [30]:
for chunk in rag_chain.stream("Summarize the document"):
    print(chunk, end="", flush=True)

The documents appear to be excerpts from the Constitution of India, specifically from Part III, which deals with Fundamental Rights. 

Document 1 seems to be related to the general structure of the Constitution, mentioning Part III and the articles 30-31A.

Document 2 discusses the restrictions on Fundamental Rights under certain circumstances. According to this document, the Parliament has the authority to restrict or abrogate Fundamental Rights for individuals working in the Armed Forces, forces maintaining public order, intelligence or counter-intelligence organizations, or telecommunication systems related to these forces or organizations, as long as it ensures the proper discharge of duties and maintenance of discipline.

Document 3 is similar to Document 2, but it seems to be an older version of Article 33, mentioning the Constitution (Fiftieth Amendment) Act, 1984, which made some changes to the article.

Overall, these documents highlight the relationship between the Fundamenta

In [31]:
    ## Query multiple questions (batch)
questions = [
    "What is the document about?",
    "What are the key points?",
    "Who is the target audience?"
]

responses = rag_chain.batch(questions)

for q, r in zip(questions, responses):
    print(f"\nQ: {q}\nA: {r}")


Q: What is the document about?
A: The documents appear to be excerpts from the Constitution of India, specifically from Part III, which deals with Fundamental Rights. 

Document 1 seems to be a provision (Article 31) about the validation of certain Acts and Regulations.

Document 2 is actually another provision (Article 31C) related to Article 13, which deals with laws that are inconsistent with the Fundamental Rights.

Document 3 is Article 15 of the Constitution of India, which prohibits the State from discriminating against citizens based on certain grounds, including religion, race, caste, sex, and place of birth.

Therefore, overall, the documents are about the Fundamental Rights enshrined in the Constitution of India, specifically about non-discrimination and the validation of certain laws.

Q: What are the key points?
A: Based on the provided context, the key points related to Fundamental Rights under The Constitution of India are:

1. **Right to Equality**: The State shall not